# Entendendo o negócio

A Zuber é uma nova empresa de compartilhamento de caronas que está sendo lançada em Chicago. Para apoiar sua entrada no mercado, este projeto analisa dados de concorrentes e informações sobre corridas, com o objetivo de identificar padrões e compreender as preferências dos passageiros.

Além de explorar os destinos e a demanda por viagens, será avaliado se fatores externos influenciam a atividade. Em particular, será testada a hipótese de que as condições climáticas afetam a frequência das corridas.

# Bibliotecas e módulos usados

In [1]:
import pandas as pd
import plotly.express as px
from scipy import stats as st
import numpy as np

# 1. Análise Exploratória de dados (EDA)

## 1.1. Importando os arquivos:

In [2]:
amount = pd.read_csv('../datasets/project_sql_result_01.csv')

In [3]:
location = pd.read_csv('../datasets/project_sql_result_04.csv')

## 1.2. Estudando uma amostra dos dados:

In [4]:
amount.sample(5)

,company_name,trips_amount
10,Choice Taxi Association,5015
37,1469 - 64126 Omar Jada,36
53,5006 - 39261 Salifu Bawa,14
5,Chicago Carriage Cab Corp,9181
32,Leonard Cab Co,147


In [5]:
location.sample(5)

,dropoff_location_name,average_trips
56,Ukrainian Village,27.600000
39,Jefferson Park,74.333333
38,Armour Square,77.033333
52,Little Village,33.800000
62,Brighton Park,19.466667


## 1.3. Verificando as informações gerais dos dados:

In [6]:
amount.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 64 entries, 0 to 63
Data columns (total 2 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   company_name  64 non-null     object
 1   trips_amount  64 non-null     int64 
dtypes: int64(1), object(1)
memory usage: 1.1+ KB


In [7]:
location.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 94 entries, 0 to 93
Data columns (total 2 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   dropoff_location_name  94 non-null     object 
 1   average_trips          94 non-null     float64
dtypes: float64(1), object(1)
memory usage: 1.6+ KB


## 1.4. Identificando os 10 principais bairros em termos de destino:

In [8]:
# ordenando location por average_trips em ordem decrescente, e selecionando as 10 primeiras linhas:
top_10_locations = location.sort_values(by = 'average_trips', ascending = False).reset_index(drop = True)
top_10_locations = top_10_locations.loc[0:9]
top_10_locations

,dropoff_location_name,average_trips
0,Loop,10727.466667
1,River North,9523.666667
2,Streeterville,6664.666667
3,West Loop,5163.666667
4,O'Hare,2546.900000
5,Lake View,2420.966667
6,Grant Park,2068.533333
7,Museum Campus,1510.000000
8,Gold Coast,1364.233333
9,Sheffield & DePaul,1259.766667


## 1.5. Plotagem de gráficos: Empresas de táxi e número de corridas, e top 10 bairros por número de corridas em que esse bairro é destino:

In [9]:
# top 10 bairros mais visitados
fig1 = px.bar(
    data_frame=top_10_locations, 
    x='dropoff_location_name', 
    y='average_trips',
    title='Top 10 Bairros com Mais Desembarques',
    labels={'dropoff_location_name': 'Bairro', 'average_trips': 'Média de Viagens'},
    color='average_trips', 
    color_continuous_scale='Viridis'
)

fig1.update_layout(xaxis_tickangle=-45)

fig1.show()

In [10]:
# Empresas de táxi e número de corridas:
fig1 = px.bar(
    data_frame = amount, 
    x='company_name', 
    y='trips_amount',
    title='Análise de Demanda: Volume de Corridas por Empresa',
    labels={
        'company_name': 'Empresa de Táxi', 
        'trips_amount': 'Total de Viagens'
    },
    color='trips_amount', 
    color_continuous_scale='Sunsetdark' 
)

# Ajustes adicionais
fig1.update_layout(
    xaxis_tickangle=-45,
    plot_bgcolor='rgba(0,0,0,0)',
    coloraxis_showscale=False      
)

fig1.show()

In [11]:
amount.describe()

,trips_amount
count,64.000000
mean,2145.484375
std,3812.310186
min,2.000000
25%,20.750000
50%,178.500000
75%,2106.500000
max,19558.000000


In [12]:
# Selecionando as top 25% empresas mais utilizadas:
top_amount = amount.sort_values(by = 'trips_amount', ascending = False).reset_index(drop = True)
mask = top_amount['trips_amount'] > 2106
top_amount = top_amount[mask]
list_top_amount = top_amount['company_name'].unique()
list_top_amount

array(['Flash Cab', 'Taxi Affiliation Services', 'Medallion Leasing',
       'Yellow Cab', 'Taxi Affiliation Service Yellow',
       'Chicago Carriage Cab Corp', 'City Service', 'Sun Taxi',
       'Star North Management LLC', 'Blue Ribbon Taxi Association Inc.',
       'Choice Taxi Association', 'Globe Taxi',
       'Dispatch Taxi Affiliation', 'Nova Taxi Affiliation Llc',
       'Patriot Taxi Dba Peace Taxi Association',
       'Checker Taxi Affiliation'], dtype=object)

## 1.6. Analisando os resultados dos gráficos:

Podemos observar que:
- Cerca de 75% das empresas fazem viagens abaixo da média (cerca de 2106 viagens no máximo, sendo a média de 2145 viagens). Um grupo seleto de empresas integram o último percentil das 25% empresas mais utilizadas, liderado pela Flash Cab. São elas: Flash Cab, Taxi Affiliation Services, Medallion Leasing, Yellow Cab, Taxi Affiliation Service Yellow, Chicago Carriage Cab Corp, City Service, Sun Taxi, Star North Management LLC, Blue Ribbon Taxi Association Inc., Choice Taxi Association, Globe Taxi, Dispatch Taxi Affiliation, Nova Taxi Affiliation Llc, Patriot Taxi Dba Peace Taxi Association, Checker Taxi Affiliation.
- O birro com mais desembarques foi o Loop, seguido por River North, Streeterville e West Loop, esses bairros foram responsáveis pela maior parte das viagens aos top 10 Bairros mais visitados.

# 2. Testando hipóteses

## 2.1. Importando os arquivos da consulta:

In [14]:
trips_loop_ohare = pd.read_csv('../datasets/project_sql_result_07.csv')

In [15]:
trips_loop_ohare.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1068 entries, 0 to 1067
Data columns (total 3 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   start_ts            1068 non-null   object 
 1   weather_conditions  1068 non-null   object 
 2   duration_seconds    1068 non-null   float64
dtypes: float64(1), object(2)
memory usage: 25.2+ KB


In [16]:
# Tratando os dados da tabela 'trips_loop_ohare':
# Convertendo 'start_ts' para o formato datetime:
trips_loop_ohare['start_ts'] = pd.to_datetime(trips_loop_ohare['start_ts'])
trips_loop_ohare.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1068 entries, 0 to 1067
Data columns (total 3 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   start_ts            1068 non-null   datetime64[ns]
 1   weather_conditions  1068 non-null   object        
 2   duration_seconds    1068 non-null   float64       
dtypes: datetime64[ns](1), float64(1), object(1)
memory usage: 25.2+ KB


In [17]:
# imprimindo uma amostra dos dados:
trips_loop_ohare.sample(5)

,start_ts,weather_conditions,duration_seconds
834,2017-11-04 16:00:00,Bad,3180.0
492,2017-11-11 15:00:00,Good,2285.0
134,2017-11-18 12:00:00,Bad,3024.0
731,2017-11-11 11:00:00,Good,1683.0
1016,2017-11-11 04:00:00,Good,1254.0


## 2.2. A duração média dos passeios do Loop para o Aeroporto Internacional O'Hare muda nos sábados chuvosos?

- Filtrando o dataframe 'trips_loop_ohare' para obter apenas os dados dos sábados chuvosos e não chuvosos:

In [18]:
mask2 = (trips_loop_ohare['weather_conditions'] == 'Bad') & (trips_loop_ohare['start_ts'].dt.weekday == 5)

mask3 = (trips_loop_ohare['weather_conditions'] == 'Good') & (trips_loop_ohare['start_ts'].dt.weekday == 5)

In [19]:
saturday_bad = trips_loop_ohare[mask2].sort_values(by = 'start_ts').reset_index(drop=True)

saturday_good = trips_loop_ohare[mask3].sort_values(by = 'start_ts').reset_index(drop=True)

### Aplicando o teste de hipóteses sobre a igualdade da média de duas populações para verificar se houve alterações na duração média dos passeios nos sábados chuvosos quando comparado à média de tempo de sábados em bos condições climáticas:

Formulando as hipóteses:
- <b>HIPÓTESE NULA</b> : A duração média dos passeios não mudou nos sábados chuvosos;
- <b>HIPÓTESE ALTERNATIVA</b> : A duração média dos passeios sofreu alterações nos sábados chuvosos;

Para esse teste, escolhi um nível de significância de 5%

In [20]:
# Vamos checar se as variâncias das duas populções são estatisticamente iguais:

# Teste de Levene para igualdade de variâncias
stat, p_levene = st.levene(saturday_bad['duration_seconds'], saturday_good['duration_seconds'])

# Se p_levene > 0.05, as variâncias são consideradas iguais
variancias_iguais = p_levene > 0.05

if p_levene > 0.05: 
    print('equal_var = True')
else: 
    print('equal_var = False')     

equal_var = True


In [21]:
alpha = 0.05 # significância estatística crítica de 5% 

# executando um teste 

results = st.ttest_ind(saturday_bad['duration_seconds'], saturday_good['duration_seconds'], equal_var = True) 
 

print('valor-p:', results.pvalue)# imprimindo o valor-p obtido 

 
# comparando o valor-p com o limiar 
if (results.pvalue < alpha): 
    print("❌ Rejeitamos a hipótese nula, isto é: A duração média dos passeios sofreu alterações nos sábados chuvosos") 
else: 
    print("✅ Não podemos rejeitar a hipótese nula, isto é: A duração média dos passeios não mudou nos sábados chuvosos") 

valor-p: 6.517970327099489e-12
❌ Rejeitamos a hipótese nula, isto é: A duração média dos passeios sofreu alterações nos sábados chuvosos


Devido tratar-se de comparar a média de duas populações, decidi pelo teste de hipóteses sobre a igualdade da média de duas populações, donde concluí que devemos rejeitar a hipótese nula. Dessa forma, aceitamos que <b> a duração média dos passeios sofreu alterações nos sábados chuvosos, estatisticamente falando <b>.